# Lekce 07 - Návrhový vzor plánování

Tento notebook demonstruje **Návrhový vzor plánování** pro AI agenty pomocí Microsoft Agent Framework.
Naučíte se, jak rozdělit složité cestovní požadavky na strukturované podúkoly, přiřadit je specialistickým agentům
a vykonat výsledný plán — vše pomocí strukturovaného výstupu podporovaného modely Pydantic.


## Nastavení


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Rozklad úkolu

Rozklad úkolu je jádrem návrhového vzoru plánování. Místo toho, abychom jednoho agenta požádali, aby komplexní požadavek vyřešil od začátku do konce, rozložíme problém na menší, dobře definované **podúkoly**. Každý podúkol je přiřazen specializovanému agentovi (např. lety, hotely, aktivity) s jasnými prioritami a pořadím závislostí.

Tento přístup přináší několik výhod:
- **Jasnost**: každý podúkol má jednu odpovědnost.
- **Paralelismus**: nezávislé podúkoly mohou běžet současně.
- **Spolehlivost**: selhání jsou izolována na jednotlivé podúkoly.
- **Sledování rozpočtu**: náklady jsou odhadovány na podúkol a shromažďovány.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Vytvoření plánovacího agenta se strukturovaným výstupem

Plánovací agent funguje jako **koordinátor na recepci**. Na základě obecné cestovní žádosti
vytvoří strukturovaný `TravelPlan` — rozloží požadavek na dílčí úkoly, nastaví priority
a identifikuje závislosti, aby mohl concierge nebo realizační vrstva práci provést.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Provádění plánu se specializovanými nástroji

Jakmile recepční vytvoří strukturovaný plán, **concierge agent** jej provede.
Každý specializovaný nástroj zpracovává jednu kategorii dílčích úkolů (letenky, hotely, aktivity). Concierge postupně prochází dílčí úkoly plánu v pořadí závislostí a předává každý z nich příslušnému nástroji.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Shrnutí

V této lekci jste se naučili **vzor návrhu plánování** pro AI agenty:

1. **Rozklad úkolu** — Plánovací agent na recepci rozdělí složitou cestovní žádost na strukturované podúkoly pomocí modelů Pydantic a přiřadí je specializovaným agentům s prioritami a závislostmi.
2. **Strukturovaný výstup** — Předáním `response_format` agent vrací validovaný objekt `TravelPlan` místo volného textu, což zajišťuje spolehlivé zpracování výsledků.
3. **Provedení plánu** — Agent concierge postupně prochází podúkoly pomocí specializovaných nástrojů (`book_flight`, `reserve_hotel`, `book_activity`), aby plán provedl a nahlásil výsledky.

Tento vzor odděluje *co dělat* (plánování) od *jak to udělat* (provedení), díky čemuž jsou agenti modulárnější, testovatelnější a snadněji rozšiřitelní.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho původním jazyce by měl být považován za závazný zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoliv nedorozumění nebo nesprávné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
